# NB2A - Pokémon Data Processing
This notebook processes Pokémon data by extracting relevant information from multiple sources, cleaning, and transforming it, and then stores the processed data in an SQLite database. This allows for future usage, analysis, integration, and visualization of the Pokémon dataset.

## Environment Setup
In this section, we import the required libraries that will be used throughout the notebook. These include:
- `json` for loading and parsing JSON files containing Pokémon data,
- `pandas` and `numpy` for data manipulation and analysis,
- `sqlalchemy` for interacting with the SQLite database,
- `requests` for fetching Pokémon images from the web,
- `sklearn` for clustering and KMeans algorithms,
- `PIL` for image processing (to extract dominant Pokémon colors),
- `concurrent.futures` for handling concurrent image processing,
- `matplotlib.colors` for classifying colors based on RGB values.

Additionally, the environment variable `OMP_NUM_THREADS` is set to `1` to optimize threading and prevent multi-threading issues during large-scale computations.


In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import json
import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text

import requests
from sklearn.cluster import KMeans
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import matplotlib.colors as mcolors

## Load Pokémon Generation Data
The Pokémon dataset containing basic information (ID, name, generation) is loaded from a JSON file (`generation_pokemon.json`). This dataset is then converted into a Pandas DataFrame (`poke_df`) for easier manipulation and further analysis. This DataFrame will serve as the central dataset for storing all processed Pokémon information.


In [3]:
with open('../../data/pokemon_data/generation_pokemon/generation_pokemon.json', 'r') as file:
    pokemon_data = json.load(file)

poke_df = pd.DataFrame(pokemon_data)
display(poke_df)

,name,url,generation,pokemon_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1,1
1,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/,1,2
2,venusaur,https://pokeapi.co/api/v2/pokemon-species/3/,1,3
3,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1,4
4,charmeleon,https://pokeapi.co/api/v2/pokemon-species/5/,1,5
...,...,...,...,...
1020,raging-bolt,https://pokeapi.co/api/v2/pokemon-species/1021/,9,1021
1021,iron-boulder,https://pokeapi.co/api/v2/pokemon-species/1022/,9,1022
1022,iron-crown,https://pokeapi.co/api/v2/pokemon-species/1023/,9,1023
1023,terapagos,https://pokeapi.co/api/v2/pokemon-species/1024/,9,1024


## Loading Sample Species and Details Data
In this step, species and Pokémon-specific details are loaded from separate JSON files for Pokémon with ID `1` to explore their attributes. These details include information like habitat and description for the species, as well as base stats and portrait URLs for the Pokémon itself. The JSON data is normalized into two separate DataFrames (`test_df_1` and `test_df_2`) to inspect their column structures and ensure that the data is in a usable format.



In [4]:
with open('../../data/pokemon_data/species_details/pokemon_1_species_details.json', 'r') as file:
    test_data = json.load(file)

test_df_1 = pd.json_normalize(test_data)
test_df_1.set_index('id', inplace = True)
display(test_df_1)
print(test_df_1.columns)

,base_happiness,capture_rate,egg_groups,evolves_from_species,form_descriptions,forms_switchable,gender_rate,genera,has_gender_differences,hatch_counter,...,flavor_text_entries.version.name,flavor_text_entries.version.url,generation.name,generation.url,growth_rate.name,growth_rate.url,habitat.name,habitat.url,shape.name,shape.url
id,,,,,,,,,,,,,,,,,,,,,
1,50,45,"[{'name': 'monster', 'url': 'https://pokeapi.c...",None,[],False,1,"[{'genus': 'たねポケモン', 'language': {'name': 'ja-...",False,20,...,red,https://pokeapi.co/api/v2/version/1/,generation-i,https://pokeapi.co/api/v2/generation/1/,medium-slow,https://pokeapi.co/api/v2/growth-rate/4/,grassland,https://pokeapi.co/api/v2/pokemon-habitat/3/,quadruped,https://pokeapi.co/api/v2/pokemon-shape/8/


Index(['base_happiness', 'capture_rate', 'egg_groups', 'evolves_from_species',
       'form_descriptions', 'forms_switchable', 'gender_rate', 'genera',
       'has_gender_differences', 'hatch_counter', 'is_baby', 'is_legendary',
       'is_mythical', 'name', 'names', 'order', 'pal_park_encounters',
       'pokedex_numbers', 'varieties', 'color.name', 'color.url',
       'evolution_chain.url', 'flavor_text_entries.flavor_text',
       'flavor_text_entries.language.name', 'flavor_text_entries.language.url',
       'flavor_text_entries.version.name', 'flavor_text_entries.version.url',
       'generation.name', 'generation.url', 'growth_rate.name',
       'growth_rate.url', 'habitat.name', 'habitat.url', 'shape.name',
       'shape.url'],
      dtype='object')


In [5]:
with open('../../data/pokemon_data/pokemon_details/pokemon_1_details.json', 'r') as file:
    test_data = json.load(file)

test_df_2 = pd.json_normalize(test_data)
test_df_2.set_index('id', inplace = True)
display(test_df_2)
print(test_df_2.columns)

,abilities,base_experience,forms,game_indices,height,held_items,is_default,location_area_encounters,moves,name,...,past_abilities,past_types,stats,types,weight,pokemon_portrait,cries.latest,cries.legacy,species.name,species.url
id,,,,,,,,,,,,,,,,,,,,,
1,"[{'ability': {'name': 'overgrow', 'url': 'http...",64,"[{'name': 'bulbasaur', 'url': 'https://pokeapi...","[{'game_index': 153, 'version': {'name': 'red'...",7,[],True,https://pokeapi.co/api/v2/pokemon/1/encounters,"[{'move': {'name': 'razor-wind', 'url': 'https...",bulbasaur,...,[],[],"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",69,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/


Index(['abilities', 'base_experience', 'forms', 'game_indices', 'height',
       'held_items', 'is_default', 'location_area_encounters', 'moves', 'name',
       'order', 'past_abilities', 'past_types', 'stats', 'types', 'weight',
       'pokemon_portrait', 'cries.latest', 'cries.legacy', 'species.name',
       'species.url'],
      dtype='object')


## Extracting Pokémon Species Data
Two key functions are used to process and extract additional data for each Pokémon:
1. **`extract_pokemon_species_data()`**: This function processes species details, extracting attributes such as:
   - Habitat Name: The environment where the Pokémon is typically found.
   - Pokémon Description: A textual flavor description of the Pokémon.
2. **`extract_pokemon_details_data()`**: This function processes individual Pokémon details, extracting:
   - Primary and Secondary Types: The types of the Pokémon (e.g., Fire, Water).
   - Base Stats: The Pokémon’s statistics, including HP, Attack, Defense, Special Attack, Special Defense, and Speed.
   - Pokémon Portrait URL: The URL for the Pokémon’s image.

These functions are applied to the dataset, enriching the DataFrame (`poke_df`) with additional columns for habitat, description, types, stats, and portrait URLs.

In [6]:
range_of_poke_id = range(poke_df['pokemon_id'].min(), poke_df['pokemon_id'].max()+1)

def extract_pokemon_species_data(): 
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/species_details/pokemon_{pokemon_id}_species_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    
    poke_df['habitat_name'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['habitat']['name'] if json_data[x]['habitat'] else None)
    poke_df['pokemon_description'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['flavor_text_entries']['flavor_text'])
    return
    
extract_pokemon_species_data()

In [7]:
poke_df['pokemon_description'] = poke_df['pokemon_description'].str.replace('\n', ' ', regex=False)

In [8]:
def extract_pokemon_details_data():
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/pokemon_details/pokemon_{pokemon_id}_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    poke_df['type_1'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][0]['type']['name'])
    poke_df['type_2'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][1]['type']['name'] if len(json_data[x]['types']) > 1 else None)
    poke_df['pokemon_portrait'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['pokemon_portrait'])

    stats = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']
    for i, stat in enumerate(stats):
        poke_df[f'{stat}_stat'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['stats'][i]['base_stat'])
        
    return 

extract_pokemon_details_data()

## Computing Total Stats
A helper function, `calculate_total_stat()`, is defined to calculate the total strength of each Pokémon by summing their individual base stats. The function identifies columns in the DataFrame related to Pokémon stats and creates a new column, `total_stat`, to represent the sum of these stats. This provides an overall measure of each Pokémon’s strength.


In [9]:
def calculate_total_stat(df: pd.DataFrame):
    stat_columns = [col for col in df.columns if 'stat' in col]
    df['total_stat'] = df[stat_columns].sum(axis=1)

calculate_total_stat(poke_df)

## Filtering Pokémon by Type 
The dataset is filtered into three separate DataFrames based on Pokémon types:
- Fire-type Pokémon (`fire_poke_df`)
- Ice-type Pokémon (`ice_poke_df`)
- Water-type Pokémon (`water_poke_df`)

Each DataFrame only contains Pokémon that match the respective type in the `type_1` column of the original `poke_df`.

In [10]:
poke_df.set_index('pokemon_id', inplace = True)
fire_poke_df = poke_df[poke_df['type_1'] == 'fire']
ice_poke_df = poke_df[poke_df['type_1'] == 'ice']
water_poke_df = poke_df[poke_df['type_1'] == 'water']

## Extracting Dominant Pokémon Colours 
The function **`get_main_colour_from_url()`** is used to extract the dominant RGB color from each Pokémon’s portrait. The steps include:
- Fetching the image using its URL,
- Converting the image to an RGBA format (to handle transparency) and then to RGB format,
- Filtering out black and white pixels to focus on the relevant color areas,
- Using KMeans clustering to determine the most prominent color in the image.

This process is carried out concurrently for multiple URLs using the `ThreadPoolExecutor` for efficient parallel processing. A known issue with Pokémon ID `678` is handled separately, as the image cannot be processed correctly, and the color is manually assigned.

In [ ]:
# 3 Things to note:
# 1. Pokémon ID 678 has a bugged image and cannot be processed correctly.
# 2. The background removal process eliminates all white pixels, meaning no Pokémon will have white as their dominant color.
# 3. A UserWarning suggests setting OMP_NUM_THREADS=2, despite it already being configured.

def get_main_colour_from_url(image_url):
    try:
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        
        img_rgba = img.convert('RGBA')
        img_rgb = img_rgba.convert('RGB')
        img_array = np.array(img_rgb)
        
        lower_black = np.array([0, 0, 0])
        upper_black = np.array([5, 5, 5])  
        
        lower_white = np.array([250, 250, 250])  
        upper_white = np.array([255, 255, 255])
    
        black_mask = np.all(np.logical_and(img_array >= lower_black, img_array <= upper_black), axis=-1)
        white_mask = np.all(np.logical_and(img_array >= lower_white, img_array <= upper_white), axis=-1)
    
        exclude_mask = black_mask | white_mask
        include_mask = ~exclude_mask
        img_filtered = img_array[include_mask].reshape(-1, 3)
        
        kmeans = KMeans(n_clusters=1, random_state=0).fit(img_filtered)
        main_colour = kmeans.cluster_centers_[0]
        return tuple(main_colour.astype(int))
    
    except Exception as e:
        print(f"Error processing {image_url}: {e}")
        return None

def get_dominant_colours_concurrently(urls):
    with ThreadPoolExecutor(max_workers=10) as executor:
        result = list(executor.map(get_main_colour_from_url, urls))
    return result


urls = poke_df['pokemon_portrait'].tolist()
pokemon_colours = get_dominant_colours_concurrently(urls)
poke_df['pokemon_colour'] = pokemon_colours


Error processing https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/678.png: cannot identify image file <_io.BytesIO object at 0x0000018A21A11FD0>


In [12]:
# To account for the missing value, I will be manually allocating the RGB value for the file that cannot be processed. 
poke_df.at[678, 'pokemon_colour'] = (63, 95, 177)

## Classifying Pokémon Colors
After extracting the dominant RGB color for each Pokémon, the **`classify_colour()`** function categorizes the color into broader color families, such as Red, Green, Blue, Yellow, etc. This classification is based on the color’s hue, saturation, and value (HSV) components. A set of predefined thresholds is used to assign Pokémon to their respective color families, allowing for easier grouping and analysis.


In [13]:
def classify_colour(rgb):
    if rgb is None:
        return 'Unknown'
    
    if len(rgb) != 3 or any(val < 0 or val > 255 for val in rgb):
        return 'Unknown'
    
    rgb_normalized = np.array(rgb) / 255.0
    hsv = mcolors.rgb_to_hsv(rgb_normalized)
    
    hue = hsv[0] * 360
    saturation = hsv[1]
    value = hsv[2]
    
    # Classify based on hue ranges and saturation/value thresholds - these hue and saturation values for classification were generated by ChatGPT 
    if saturation < 0.2:  
        return 'Gray' if value > 0.5 else 'Brown'
    elif 0 <= hue < 30 or 330 <= hue < 360:
        return 'Red'
    elif 30 <= hue < 60:
        return 'Orange'
    elif 60 <= hue < 120:
        return 'Yellow'
    elif 120 <= hue < 180:
        return 'Green'
    elif 180 <= hue < 240:
        return 'Cyan'
    elif 240 <= hue < 300:
        return 'Blue'
    elif 300 <= hue < 330:
        return 'Purple'
    elif 330 <= hue < 360:
        return 'Pink'
    else:
        return 'Unknown'  
    
poke_df['colour_family'] = poke_df['pokemon_colour'].apply(classify_colour)


In [14]:
print(poke_df['pokemon_colour'].value_counts())
print(poke_df['colour_family'].value_counts())

pokemon_colour
(166, 151, 177)    2
(161, 117, 85)     1
(84, 128, 70)      1
(122, 128, 122)    1
(92, 122, 144)     1
                  ..
(86, 88, 79)       1
(105, 70, 74)      1
(115, 78, 76)      1
(148, 135, 81)     1
(90, 61, 99)       1
Name: count, Length: 1024, dtype: int64
colour_family
Red       252
Orange    166
Cyan      146
Gray      146
Yellow     97
Brown      96
Blue       56
Purple     34
Green      32
Name: count, dtype: int64


## Saving Data to JSON
Once all the data processing steps have been completed, the enriched DataFrame (`poke_df`) is saved as a JSON file (`main_pokemon_df.json`). This allows for the processed dataset to be accessed and used in future projects, ensuring that the data is stored in a standardized and reusable format.

In [15]:
poke_df.to_json('../../data/pokemon_data/main_pokemon_df.json')

## Ranking Pokémon by Strength
A function **`assign_locations()`** is used to assign a ranking to each Pokémon based on their overall strength. The Pokémon are sorted by their `total_stat`, with ties broken by the `hp_stat`. The function ranks the Pokémon within each type category (Fire, Ice, Water) and assigns each Pokémon a rank based on their performance.


In [16]:
def assign_locations(df):
    df = df.copy()
    df.sort_values(by=['total_stat', 'hp_stat'], ascending=[False, False], inplace=True)
    df.loc[:,'ranking'] = range(1, len(df) + 1)
    return df

In [17]:
fire_poke_df = assign_locations(fire_poke_df)
ice_poke_df = assign_locations(ice_poke_df)
water_poke_df = assign_locations(water_poke_df)

## **Storing Data in an SQLite Database**
The processed Pokémon data is then stored in an SQLite database (`main.db`). Three separate tables are created for each Pokémon type: `fire_pokemon`, `ice_pokemon`, and `water_pokemon`. Each table includes the following attributes:
- Pokémon ID, Name, and Generation
- Type and Habitat information
- Base Stats, Total Stats, and Ranking
- Dominant Color and Color Family

These tables are created using SQLAlchemy’s `create_engine` and `text` functions. The processed DataFrames are inserted into their respective tables using the `to_sql()` method.

### **Final Verification**
Finally, a query is executed to verify the number of rows in each table within the database. This ensures that the data has been correctly stored and provides an easy way to verify the completeness of the process.

In [19]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS fire_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY,
    pokemon_colour VARCHAR(20), 
    colour_family VARCHAR(20)                           
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS fire_pokemon;'))
    conn.execute(create_tracks_statement)

fire_poke_df.to_sql("fire_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM fire_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 66


In [20]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS ice_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),
    generation VARCHAR(1),
    habitat_name VARCHAR(20), 
    pokemon_portrait VARCHAR(50),      
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY,
    pokemon_colour VARCHAR(20), 
    colour_family VARCHAR(20)                           
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS ice_pokemon;'))
    conn.execute(create_tracks_statement)

ice_poke_df.to_sql("ice_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM ice_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 31


In [21]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS water_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY,
    pokemon_colour VARCHAR(20), 
    colour_family VARCHAR(20)                           
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS water_pokemon;'))
    conn.execute(create_tracks_statement)

water_poke_df.to_sql("water_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM water_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 134
